In [43]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [44]:
# import train dataset
train = pd.read_csv('train.csv')
train.head()

,Unnamed: 0,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,1,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,2,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,3,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,4,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,5,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [45]:
# TRAIN:
# ignore columns: id, date and zipcode
# also ignore price column because this is the response and what we are trying to predict
columns_to_drop = ['Unnamed: 0', 'zipcode', 'price']
x_train = train.drop(columns=columns_to_drop)
x_train.head()

,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,lat,long,sqft_living15,sqft_lot15
0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,47.5112,-122.257,1340,5650
1,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,47.7210,-122.319,1690,7639
2,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,47.7379,-122.233,2720,8062
3,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,47.5208,-122.393,1360,5000
4,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,47.6168,-122.045,1800,7503


In [46]:
# scale the data so that each feature has a mean of 0 and standard deviation of 1
scaler = StandardScaler()

# scale train dataset
for col in x_train:
    train_scaled = x_train[col].values.reshape(-1, 1) #reshape
    x_train[col] = scaler.fit_transform(train_scaled) 


# check mean and standard deviation of each feature to double check that scaling worked
for feature in x_train:
    mean = x_train.mean()
    std = x_train.var()


# organize all variables into a table
stats_train = {
    'mean': mean,
    'std': std
}


feature_data_train = pd.DataFrame(stats_train)
feature_data_train


,mean,std
bedrooms,-1.936229e-16,1.001001
bathrooms,5.329071e-17,1.001001
sqft_living,8.881784e-17,1.001001
sqft_lot,3.730349e-17,1.001001
floors,2.433609e-16,1.001001
waterfront,1.065814e-17,1.001001
view,3.463896e-17,1.001001
condition,8.526513e-17,1.001001
grade,6.394885e-17,1.001001
sqft_above,-1.048051e-16,1.001001


In [32]:
# divide the price by 1000 for all rows in the train dataset
y_train = train['price'] / 1000

In [47]:
# convert to matrices
#matrix_train = x_train.values

#price_matrix_train = y_train.values


In [48]:
# include a column of ones into features for the bias
# lecture slides showed column of ones in features matrix:
shape = x_train.shape[0]
x_train = np.append(x_train, np.ones((shape,1)), axis=1)

1. Write code for gradient descent for training linear regression using the algorithm from class.

In [49]:
# gradient descent for training linear regression
# algorithm: thetaj <- thetaj - alpha*deritivative of J(theta) - alpha is learning rate (small)
# algorithm should stop when: 1. update in theta is below some threshold or 2. maximum number of iterations reached (max_iterations var)
# choose initial value for theta - choose new values for theta to reduce J(theta) -> theta should get closer to minimum?

# gradient = slope of line tangent to curve

# features = x, price = y (both matrices?)
def gradient_descent(features, price, alpha, max_iterations):
    
    # initialize theta to 0?
    # length of theta vector should be amt of features + 1?
    theta = np.zeros(features.shape[1])
    N = features.shape[0]

    for iteration in range(max_iterations):
        # 1. get predicted values
        y_pred = features @ theta #?

        # 2. calc error difference btwn predicted and actual?
        error = y_pred - price

        # 3. gradient descent?
        gradient = (1/N) * (features.T @ error)

        # 4. update:
        theta = theta - (alpha * gradient)

    return theta

2. Vary the value of the learning rate (at least 3 different values $\alpha \in \{0.01,0.1,0.5\}$) and report the value of the model parameter $\theta$ after different number of iterations (10, 50, and 100). Include in a table the MSE and $R^2$ metrics on the training and testing set for the different number of iterations and different learning rates. You can choose more values of the learning rates to observe how the
behavior of the algorithm changes.

In [50]:
grad_desc = gradient_descent(x_train, y_train, 0.1, 100)
grad_desc

array([-12.44526791,  17.60656956,  57.19374274,   7.81775598,
         8.00813971,  63.41760706,  47.9672335 ,  13.64016111,
        87.80063438,  48.7904438 ,  27.1364244 , -64.92117088,
        18.12801981,  79.07064343,  -2.60731284,  49.02379265,
       -10.10832359, 520.40101105])

In [37]:
def predict_response(x, theta):
    return x @ theta

pred = predict_response(x_train, grad_desc)

In [38]:
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error


MSE_train = mean_squared_error(y_train, pred)
r2_train = r2_score(y_train, pred)

print('MSE metric: ', MSE_train)
print('R squared metric: ', r2_train)

MSE metric:  31497.692325988133
R squared metric:  0.7264333377844296


3. Write some observations about the behavior of the algorithm: How do the metrics change with different learning rates;How many
iterations are needed; Does the algorithm converge to the optimal solution, etc.